# Analyse exploratoire des données (EDA)

Jeu de données **Credit Card Fraud Detection** (Kaggle).
Objectif : comprendre la structure, le déséquilibre des classes et
les relations entre les features avant de modéliser.


In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ajout du dossier src au chemin pour réutiliser nos fonctions
sys.path.insert(0, os.path.abspath('../src'))
from preprocessing import load_data

sns.set_theme(style='whitegrid')
%matplotlib inline


## 1. Chargement des données


In [ ]:
df = load_data('data/creditcard.csv')
df.head()


In [ ]:
print(f'Nombre de transactions : {df.shape[0]:,}')
print(f'Nombre de colonnes     : {df.shape[1]}\n')
df.info()


## 2. Statistiques descriptives


In [ ]:
df.describe()


## 3. Distribution des classes

Le jeu de données est **extrêmement déséquilibré** : attendu ~0,17 % de fraudes.


In [ ]:
counts = df['Class'].value_counts()
print(counts)
print(f"Pourcentage de fraudes : {counts[1] / counts.sum() * 100:.4f} %")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Class', ax=ax)
ax.set_xticklabels(['Normale (0)', 'Fraude (1)'])
ax.set_title('Distribution des classes')
plt.show()


## 4. Analyse des montants


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df['Class'] == 0]['Amount'], bins=50, ax=axes[0], log_scale=True)
axes[0].set_title('Montant - transactions normales (échelle log)')
sns.histplot(df[df['Class'] == 1]['Amount'], bins=50, ax=axes[1], log_scale=True)
axes[1].set_title('Montant - fraudes (échelle log)')
plt.tight_layout()
plt.show()


In [ ]:
df.groupby('Class')['Amount'].describe()


## 5. Analyse temporelle


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.histplot(df[df['Class'] == 0]['Time'], bins=50, ax=axes[0])
axes[0].set_title('Time - transactions normales')
sns.histplot(df[df['Class'] == 1]['Time'], bins=50, ax=axes[1])
axes[1].set_title('Time - fraudes')
plt.tight_layout()
plt.show()


## 6. Corrélations avec la cible


In [ ]:
corr = df.drop(columns=['Class']).corrwith(df['Class']).sort_values()
corr.plot(kind='barh', figsize=(8, 10))
plt.title('Corrélation de chaque feature avec Class')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 12))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm', center=0, cbar=False)
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()


## 7. Conclusion

- Classe fortement déséquilibrée : **il faudra des métriques orientées**
  **classes minoritaires** (Rappel, Precision, PR-AUC) et une gestion
  du déséquilibre (class_weight / SMOTE).
- Les montants sont très variables → **standardisation** nécessaire.
- Les composantes V1-V28 sont déjà centrées-réduites.
- `Time` présente des motifs, signe d'une dépendance temporelle.
